<!-- cabecera-entorno -->
## Antes de empezar

**Clase 3 · Limpieza de datos** — Bloque 3 · Reto. Este cuaderno lo recorre **usted solo**, leyendo: cada tarea trae la explicación y los comandos que necesita. El profesor circula por el salón resolviendo dudas. Es el entregable de la clase.

**La rutina de siempre:** `git pull` antes de clase, y el entorno virtual activo (`(.venv)` en la
terminal). Si va a modificar este archivo, trabaje sobre una copia: duplique `reto.ipynb` como
`reto_mio.ipynb` y edite el duplicado. Así `git pull` nunca le reclama.

**Si la celda de abajo falla, no siga:** la respuesta está en el manual del entorno,
[`../INSTALACION.md`](../INSTALACION.md).

| Si ve esto | Qué pasó | Dónde se arregla |
|------------|----------|------------------|
| `ModuleNotFoundError` | El entorno virtual no está activo, o VSCode eligió otro intérprete | Manual, secciones 6.3 y 8.4, y problema 5 |
| `FileNotFoundError` al leer el CSV | El cuaderno se abrió desde otra carpeta, o falta hacer `git pull` | Manual, problema 6 |
| El kernel no aparece en VSCode | Falta la extensión Jupyter o `ipykernel` dentro del entorno | Manual, problema 4 |

In [ ]:
# Verificación del entorno. Si algo falla aquí, la solución está en ../INSTALACION.md
import sys
from pathlib import Path

try:
    import pandas as pd
    import numpy as np
except ModuleNotFoundError as error:
    raise ModuleNotFoundError(
        f"Falta la librería '{error.name}'. Active el entorno virtual y seleccione el intérprete "
        ".venv en VSCode (Ctrl+Shift+P > Python: Select Interpreter), luego reinicie el kernel. "
        "Ver ../INSTALACION.md, problema 5."
    ) from error

print("Intérprete:", sys.executable)
if ".venv" not in sys.executable:
    print("AVISO: este no parece el Python del entorno virtual. En VSCode: Ctrl+Shift+P >",
          "'Python: Select Interpreter' > el que dice .venv, y reinicie el kernel.")

RUTA_VERIFICACION = "../datos/indicadores_salud.csv"
if Path(RUTA_VERIFICACION).exists():
    print("Datos: encontrados en", RUTA_VERIFICACION)
else:
    print("FALTA el archivo", RUTA_VERIFICACION, "- abra en VSCode la carpeta raíz del curso",
          "y ejecute 'git pull'. Ver ../INSTALACION.md, problema 6.")

# Clase 3 · Reto — Limpiar indicadores de salud materno-infantil

**Dataset:** `../datos/indicadores_salud.csv` (Ministerio de Salud, datos.gov.co)
**Consigna completa:** `README.md`

532 filas x 15 columnas. Una fila = un municipio en un año, entre 2005 y 2020.

Los mismos 5 problemas del demo, en otro dataset. **El workflow se copia; la lista de columnas no.**

| Columna | Qué es | Unidad |
|---------|--------|--------|
| `cod_departamento` | Código DANE del departamento | entero |
| `departamento` | Nombre del departamento | texto |
| `cod_municipio` | Código DANE del municipio | texto (debería ser entero) |
| `municipio` | Nombre del municipio | texto |
| `ano` | Año | decimal (debería ser entero) |
| `bajo_peso_nacer` | Nacidos con bajo peso | **porcentaje (0-100)** |
| `controles_prenatales` | Promedio de controles por gestante | número de controles |
| `fecundidad_adolescente` | Fecundidad adolescente | tasa por mil |
| `mortalidad_fetal` | Mortalidad fetal | tasa por mil |
| `mortalidad_general` | Mortalidad general | tasa por mil |
| `mortalidad_infantil` | Mortalidad en menores de 1 año | tasa por mil |
| `mortalidad_materna` | Mortalidad materna | tasa por cien mil |
| `mortalidad_neonatal` | Mortalidad neonatal | tasa por mil |
| `partos_cesarea` | Partos por cesárea | **porcentaje (0-100)** |
| `partos_institucionales` | Partos atendidos en institución | **porcentaje (0-100)** |

**Tres cosas que hay que saber antes de escribir la primera línea:**

1. **La ruta sube un solo nivel** (`../datos/`), no dos como en el demo. El dataset del demo es
   compartido y vive en la raíz; este es propio de la clase.
2. **Solo tres columnas son porcentajes** acotados entre 0 y 100: `bajo_peso_nacer`,
   `partos_cesarea` y `partos_institucionales`. Las de mortalidad son tasas por mil o por cien mil
   y **pueden pasar de 100 sin que haya ningún error**. Esta es la trampa del reto.
3. **`municipio` no se limpia.** Compruébelo usted mismo en la parte 5: ya está consistente. No toda
   columna de texto está sucia, y verificarlo es parte del trabajo.

## Cómo se recorre este cuaderno

Usted trabaja solo. Nadie va a dictar los pasos desde el tablero, así que cada tarea trae todo lo
que necesita para resolverse leyendo:

| Parte de la tarea | Qué contiene |
|-------------------|--------------|
| **La pregunta** | Lo que hay que responder, escrito en español |
| **El concepto** | Qué técnica aplica y por qué esa y no otra |
| **Los comandos** | Las instrucciones que va a usar, escritas de forma genérica |
| **Lo que decide usted** | Qué columna, qué umbral, qué orden. Ahí no hay respuesta escrita |
| **La celda de código** | Los pasos numerados en comentarios. Usted escribe las líneas |
| **La comprobación** | `comprobar('TN', ...)` le dice si el resultado es correcto, sin mostrárselo |

**Por qué esto sigue siendo un reto y no una copia.** En el demo limpió estadísticas de educación.
Aquí hay indicadores de salud que no ha visto nunca, con otras columnas, otras unidades y una trampa
de dominio adentro. Le damos el camino —los comandos, la técnica—, pero el camino lo recorre usted:
elige qué columna entra en cada lista, decide qué hacer con lo que falta, distingue un porcentaje de
una tasa por mil, y **escribe por qué**. La técnica se guía; el criterio no se guía, y el criterio es
lo que se evalúa.

**Las celdas `comprobar(...)`** comparan una huella digital de su resultado con la esperada. Si
coinciden dicen `CORRECTO`; si no, dan una pista dirigida al error más probable. Nunca muestran la
respuesta.

**Las celdas de justificación** (*Tu decisión:*, *Tu respuesta:*) no llevan código, y son las que más
pesan. En limpieza de datos, un cuaderno con las once tareas en verde y ninguna justificación escrita
está a medias: el código lo escribe cualquiera, la decisión la defiende usted.

**Este cuaderno se ejecuta en orden y una sola vez.** Las tareas modifican `df` una encima de la
otra, así que ejecutar dos veces la misma celda cambia los números. Si se enreda: Kernel → Restart
and Run All, y vuelva a bajar.

**Al final** hay un punto de control que le dice cuántas de las once tareas quedaron correctas.

---

## Paso 0 · Cargar y reconocer

**El concepto.** `pd.read_csv()` deja el archivo en memoria como DataFrame. `df.copy()` guarda una
copia independiente del original: es el término de comparación de la parte 7, y sin ella no hay
antes contra el cual comparar. Ojo: `df_original = df` **no** copia nada, deja dos nombres apuntando
a la misma tabla.

**Los comandos.**

```python
df = pd.read_csv('ruta/al/archivo.csv')
df_original = df.copy()
```

Esta celda ya está escrita. Ejecútela y confirme que dice 532 filas y 15 columnas.

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv('../datos/indicadores_salud.csv')
df_original = df.copy()

print("Filas:", df.shape[0])
print("Columnas:", df.shape[1])

In [ ]:
# Verificador de las tareas. Ejecute esta celda una vez y siga adelante.
# No hace falta entenderla hoy: es andamiaje del curso, no materia de la clase.
import hashlib

_RESULTADOS = {}

_PISTAS = {
    "T1": "Son dos numeros. El primero es cuantas COLUMNAS tienen al menos un nulo, no cuantas celdas vacias hay. El segundo es el porcentaje mas alto, redondeado a un decimal: porcentaje sobre el total de filas del dataset, no sobre las filas de esa columna.",
    "T2": "Se comprueba cuantas filas le quedan a df. Use dropna(subset=['departamento']): sin subset borra toda fila que tenga algun nulo en cualquier columna y se lleva medio dataset. Y ejecutela una sola vez.",
    "T3": "Se comprueban dos cosas: que no perdio filas y que no le queda ningun nulo. Si le quedan nulos, falto alguna de las seis columnas de la lista. Si perdio filas, uso dropna en vez de fillna. La mediana se calcula columna por columna, no una sola para todas.",
    "T4": "Se comprueban el dtype de 'ano' (tiene que quedar en int64) y la suma de la columna. Si el dtype sigue en float64, falta el astype(int). Si la suma no cuadra, el fillna se aplico con un valor que no era 0 o se ejecuto la celda dos veces.",
    "T5": "Son dos cosas: cuantos NaN aparecieron al convertir (esos son los 'sin dato') y el dtype final, que debe ser int64. Quite las comas ANTES de to_numeric, o los codigos con coma tambien se le vuelven NaN y el conteo se dispara.",
    "T6": "Son dos numeros: cuantos duplicados habia y cuantas filas quedaron. Cuente ANTES de eliminar. Si le da 0 duplicados, ya ejecuto la celda una vez; vuelva a correr el cuaderno desde el principio.",
    "T7": "Son dos numeros: valores unicos antes y despues de estandarizar 'departamento'. Guarde el conteo de antes en una variable ANTES de tocar la columna. La receta completa es strip, upper, quitar tildes y quitar la puntuacion; si le quedan mas de los esperados, falta alguno de los cuatro.",
    "T8": "Son dos numeros: cuantos duplicados aparecieron despues de estandarizar el texto y cuantas filas quedaron. Si le da 0 duplicados nuevos, es que estandarizo el texto antes de la tarea 6 o que ya elimino estos.",
    "T9": "Es un solo numero: cuantos valores estan fuera del rango 0-100, sumando las TRES columnas que son porcentajes. Si le da un numero grande, esta contando tambien las columnas de mortalidad, que son tasas por mil y no se validan contra 100.",
    "T10": "Son dos numeros: cuantos valores invalidos quedan (tiene que ser 0) y cuantas filas tiene df. Si perdio filas, elimino en vez de corregir: la decision era reemplazar el valor, no borrar la fila.",
    "T11": "Es una lista de cinco nombres de municipio, no una tabla. Ordene por 'mortalidad_infantil' con ascending=False, tome head(5) y saque la columna 'municipio' con .tolist(). Si no coincide, revise que no haya modificado la columna 'municipio': esa columna no se estandariza en este reto."
}

_ESPERADO = {
    "T1": "d176e9280e",
    "T2": "d1d49300e5",
    "T3": "b3c635bcb7",
    "T4": "58f2f0083a",
    "T5": "285ce8db64",
    "T6": "09a5dbd0b0",
    "T7": "7aa38ce9c9",
    "T8": "b0164e77bc",
    "T9": "29626791d0",
    "T10": "505ee565e2",
    "T11": "8bc024e896"
}


def _firma(valor):
    """Reduce un resultado a un texto reproducible, sin importar como se calculo."""
    if isinstance(valor, (list, tuple)):
        return "lista|" + "|".join(_firma(v) for v in valor)
    if isinstance(valor, pd.DataFrame):
        partes = ["DataFrame", str(valor.shape), str([str(c) for c in valor.columns]),
                  str([str(i) for i in valor.index])]
        for columna in valor.columns:
            serie = valor[columna]
            if pd.api.types.is_bool_dtype(serie) or not pd.api.types.is_numeric_dtype(serie):
                partes.append(f"{columna}:{[str(v) for v in serie.tolist()]}")
            else:
                partes.append(f"{columna}:{round(float(serie.sum()), 4)}")
        return "|".join(partes)
    if isinstance(valor, pd.Series):
        return "|".join(["Series", str(len(valor)), str([str(i) for i in valor.index]),
                         str([str(v) for v in valor.tolist()])])
    if not isinstance(valor, str):
        try:
            return f"numero|{round(float(valor), 4)}"
        except (TypeError, ValueError):
            pass
    return f"otro|{valor!r}"


def _huella(valor):
    return hashlib.sha256(_firma(valor).encode("utf-8")).hexdigest()[:10]


def comprobar(clave, valor):
    """Dice si el resultado es el correcto, sin revelar cual era."""
    _RESULTADOS[clave] = False
    if valor is None:
        print(f"[{clave}] Sin resolver todavia: la variable sigue valiendo None.")
        return
    if isinstance(valor, pd.DataFrame):
        print(f"[{clave}] Usted produjo un DataFrame de {valor.shape[0]} filas "
              f"y {valor.shape[1]} columnas.")
    elif isinstance(valor, pd.Series):
        print(f"[{clave}] Usted produjo una Series de {len(valor)} elementos, tipo {valor.dtype}.")
    elif isinstance(valor, (list, tuple)):
        print(f"[{clave}] Usted produjo: {list(valor)}")
    else:
        print(f"[{clave}] Usted produjo: {valor!r}")
    if _huella(valor) == _ESPERADO[clave]:
        _RESULTADOS[clave] = True
        print(f"[{clave}] CORRECTO.")
    else:
        print(f"[{clave}] Todavia no coincide.")
        print(f"[{clave}] Pista: {_PISTAS[clave]}")


def resumen_puntos_de_control():
    """Estado de las tareas."""
    orden = ['T1', 'T2', 'T3', 'T4', 'T5', 'T6', 'T7', 'T8', 'T9', 'T10', 'T11']
    print("Punto de control")
    print("-" * 46)
    for clave in orden:
        estado = "correcta" if _RESULTADOS.get(clave) else "pendiente"
        print(f"  {clave}: {estado}")
    logradas = sum(1 for c in orden if _RESULTADOS.get(c))
    print("-" * 46)
    print(f"{logradas} de {len(orden)} correctas.")


print("Verificador listo. Se comprueba con comprobar('T1', su_variable).")


---

## Parte 1 · Inspección

**Qué se practica aquí.** El paso 1 del workflow: **entender antes de tocar**. Es vaciar la bolsa de
la lavandería y mirar antes de encender la lavadora.

El ritual son cinco comandos, siempre en el mismo orden: `shape`, `head()`, `dtypes`,
`isnull().sum()` y `describe()`. Los tres primeros están escritos abajo; los otros dos son la tarea 1.

**No cierre estas salidas:** de aquí salen los nombres de columna exactos y las sospechas que va a
confirmar en el resto del cuaderno.

In [ ]:
print("Forma:", df.shape)
print()
print("Columnas:", df.columns.tolist())
print()
print("Tipos:")
print(df.dtypes.to_string())

In [ ]:
df.head()

### Tarea 1 · Cuantificar lo que falta

**La pregunta.** ¿Cuántas columnas tienen al menos un valor nulo, y cuál es el **porcentaje** de
nulos de la peor de ellas?

**El concepto.** `isnull()` devuelve un DataFrame del mismo tamaño lleno de `True`/`False`, y
`.sum()` los cuenta por columna: es el mismo truco de la máscara booleana de la clase 2, donde
`True` vale 1. El número absoluto no sirve para decidir nada: **el porcentaje sí**. 60 nulos es un
desastre en 100 filas y una anécdota en 100.000, y el marco de decisión de la parte 2 se aplica
sobre porcentajes, no sobre conteos.

**Los comandos.**

```python
nulos = df.isnull().sum()          # conteo por columna
nulos[nulos > 0]                   # solo las columnas afectadas
(nulos > 0).sum()                  # cuantas columnas estan afectadas
(df.isnull().sum() / len(df) * 100).round(1)   # el porcentaje
```

**Lo que decide usted.** Cuál es el denominador del porcentaje (¿el total de filas del dataset, o
las filas de esa columna?) y cómo se cuenta una columna afectada frente a una celda vacía. Son dos
preguntas distintas y dan dos números distintos.

In [ ]:
# TU CÓDIGO AQUÍ
# 1. Cuente los nulos por columna y quédese solo con las que tengan alguno.
# 2. Guarde en n_columnas_con_nulos cuántas columnas están afectadas.
# 3. Calcule el porcentaje de nulos por columna, redondeado a 1 decimal, y muéstrelo
#    ordenado de mayor a menor.
# 4. Guarde en pct_peor el porcentaje más alto.
# 5. Muestre también describe() y mire con atención las filas min y max.

n_columnas_con_nulos = None
pct_peor = None

In [ ]:
if n_columnas_con_nulos is None or pct_peor is None:
    comprobar('T1', None)
else:
    comprobar('T1', [n_columnas_con_nulos, pct_peor])

### Su diagnóstico

**Esto es lo que se entrega, no los comandos.** Antes de escribir una sola línea de limpieza, deje
por escrito qué encontró: los 5 problemas, dónde está cada uno y con qué comando lo detectó. Las
casillas que todavía no puede llenar (duplicados, texto) se dejan en "por confirmar" y se completan
al pasar por su parte.

| # | Problema | Dónde lo vio | Comando |
|---|----------|--------------|---------|
| 1 | Nulos | | |
| 2 | Tipos incorrectos | | |
| 3 | Duplicados | | |
| 4 | Inconsistencias de texto | | |
| 5 | Valores inválidos | | |

*Tu diagnóstico, en dos o tres frases:*

---

## Parte 2 · Valores nulos

**Qué se practica aquí.** El marco de decisión. No es un comando, es un **criterio**, y es lo que se
evalúa en el Momento 1.

| Nulos en la columna | Acción | Por qué |
|---------------------|--------|---------|
| Más del 50% | Considere eliminar la columna | Más huecos que datos: lo que rellene es invento |
| Menos del 5% | Elimine las filas afectadas | Pierde poquísimo y no inventa nada |
| Entre 5% y 50% | Rellene, con criterio de dominio | Perder tantas filas es demasiado; hay que estimar |

**Pista honesta:** en este dataset **ninguna columna pasa del 50%**. Si concluyó que hay que eliminar
una columna entera, revise la cuenta de la tarea 1. Es una diferencia deliberada con el demo: el
marco se aplica igual, pero no todas sus filas se activan siempre.

### Sus decisiones, antes de programarlas

Una línea por columna afectada: qué va a hacer y **por qué**. Escribirlo después de ejecutar el
código no es justificar, es narrar.

*Tus decisiones:*

- `departamento` (3,0%): ...
- Columnas de tasas (8% a 13%): ...

### Tarea 2 · Las filas sin departamento

**La pregunta.** ¿Cuántas filas le quedan al DataFrame después de eliminar las que no tienen
departamento?

**El concepto.** `departamento` es un identificador, no una medición: una fila sin departamento es
una carta sin dirección, no se puede ubicar ni agrupar ni adivinar. Está por debajo del 5%, así que
el marco dice eliminar las filas.

`dropna(subset=['columna'])` elimina solo las filas donde **esa** columna es nula. El parámetro
`subset` es todo: `dropna()` a secas elimina toda fila que tenga algún nulo en **cualquier** columna,
y aquí eso se llevaría medio dataset sin decir una palabra.

**Los comandos.**

```python
len(df.dropna())                        # el desastre, solo para verlo
df = df.dropna(subset=['columna'])      # lo que si queremos
```

**Lo que decide usted.** Qué columna va en `subset`, y comprobar de paso cuánto se habría llevado un
`dropna()` sin él. Ejecute la celda **una sola vez**.

In [ ]:
# TU CÓDIGO AQUÍ
# 1. Imprima cuántas filas quedarían con dropna() sin subset. No lo asigne: es solo para verlo.
# 2. Elimine con dropna(subset=[...]) las filas sin departamento y reasigne df.
# 3. Imprima cuántas filas había, cuántas quedaron y cuántas se eliminaron.

In [ ]:
comprobar('T2', len(df))

### Tarea 3 · Rellenar las tasas

**La pregunta.** Rellene con la **mediana de cada columna** los nulos de las seis columnas de
indicadores. ¿Le quedan filas completas y cero celdas vacías?

**El concepto.** Están entre 5% y 50%: hay que estimar. Se rellena con la **mediana** —el valor del
medio al ordenar los datos— y no con el promedio, porque el promedio se deja arrastrar por los
valores extremos y la mediana no. Este dataset tiene una `mortalidad_materna` de más de 400 y una
`partos_cesarea` de 141: si rellenara con el promedio, esos valores se le meterían en todas las filas
que rellene. Es la analogía del salario del CEO.

Y no se rellena con 0, porque una mortalidad infantil de 0 afirma que no murió ningún niño. `NaN` es
**desconocido**; 0 es **un dato**. No son lo mismo, y la diferencia se arrastra hasta la conclusión.

**Ojo con una cosa:** la mediana se calcula **columna por columna**. Una sola mediana para todas
mezclaría porcentajes con tasas por cien mil.

**Los comandos.**

```python
mediana = df['columna'].median()
df['columna'] = df['columna'].fillna(mediana)

for col in lista_de_columnas:      # el mismo bloque, repetido
    ...
```

**Lo que decide usted.** **Cuáles** son las columnas de la lista. No las copie del demo: aquí no
existe `desercion`. Sáquelas de la salida de la tarea 1, que es donde están las que tienen nulos.

In [ ]:
# TU CÓDIGO AQUÍ
# 1. Arme la lista de las columnas que todavía tienen nulos (sáquela de la tarea 1).
# 2. Recorra la lista y rellene cada columna con su propia mediana.
# 3. Verifique: imprima cuántas celdas vacías quedan en todo el DataFrame.

In [ ]:
comprobar('T3', [len(df), int(df.isnull().sum().sum())])

---

## Parte 3 · Tipos de datos

**Qué se practica aquí.** El zapato en la pila de camisas: un número guardado como otra cosa. Dos
columnas están mal tipadas y por razones distintas.

- `ano` es decimal (`2011.0`) y debería ser entero.
- `cod_municipio` es texto: algunos valores traen coma como separador de miles (`44,001`) y otros
  dicen literalmente `sin dato`.

**La regla de orden, que es la que se olvida: rellenar primero, convertir después.** `astype(int)`
revienta si hay `NaN`, y la razón es conceptual: `NaN` es un decimal y no existe ningún entero que
signifique "desconocido".

### Tarea 4 · Arreglar `ano`

**La pregunta.** Deje `ano` como entero, sin perder ningún valor.

**El concepto.** La secuencia segura tiene tres partes, y cada una resuelve un problema distinto:

1. `pd.to_numeric(serie, errors='coerce')` convierte a número y **lo que no pueda convertir lo vuelve
   `NaN` en vez de reventar**. Ese parámetro convierte un error fatal en información: ahora usted
   puede contar cuántos casos problemáticos hay.
2. `fillna(...)` elimina los `NaN`, con la decisión que corresponda.
3. `astype(int)` ya puede convertir sin riesgo.

**Los comandos.**

```python
df['columna'] = pd.to_numeric(df['columna'], errors='coerce').fillna(0).astype(int)
df['columna'].dtype
df['columna'].head(5).tolist()
```

**Lo que decide usted.** Con qué rellenar antes de convertir, y si le parece bien lo que eso implica.
La comprobación mira el `dtype` (tiene que quedar en `int64`) y la **suma** de la columna, que se
mueve si rellenó con algo distinto.

In [ ]:
# TU CÓDIGO AQUÍ
# 1. Imprima el dtype y una muestra de 'ano' antes de tocarla.
# 2. Conviértala a entero con la secuencia segura.
# 3. Imprima el dtype y la muestra después.

In [ ]:
comprobar('T4', (str(df['ano'].dtype), int(df['ano'].sum())))

### Tarea 5 · Arreglar `cod_municipio`

**La pregunta.** ¿Cuántos valores de `cod_municipio` **no** se pueden convertir a número? Déjela como
entero.

**El concepto.** Mismo problema que la columna `poblacion_5_16` del demo, con los mismos dos
culpables: la coma de miles y un literal de texto. Cualquiera de los dos basta para que pandas
guarde la columna entera como texto; ante la duda, pandas no adivina.

**El orden importa y aquí sí produce un número equivocado:** si hace `to_numeric` **antes** de quitar
las comas, los códigos que traen coma tampoco se pueden convertir y también se le vuelven `NaN`. El
conteo de "sin dato" se le dispara y usted no se entera.

**Los comandos.**

```python
df['columna'].unique()[:20]                                  # ver el problema primero
df['columna'].astype(str).str.replace(',', '', regex=False)  # quitar las comas
pd.to_numeric(serie, errors='coerce')                        # convertir; lo que no, NaN
serie.isnull().sum()                                         # contar lo que no se pudo
serie.fillna(0).astype(int)                                  # cerrar
```

**Lo que decide usted.** El orden de los dos primeros pasos, y qué hacer con los códigos que no se
pueden recuperar. Un código DANE 0 no existe: es una decisión discutible y por eso hay una celda
abajo para defenderla.

In [ ]:
# TU CÓDIGO AQUÍ
# 1. Muestre unos 20 valores de la columna para ver por qué es texto.
# 2. Quite las comas.
# 3. Convierta con to_numeric(errors='coerce') y guarde en n_sin_dato cuántos NaN aparecieron.
# 4. Rellene con 0, convierta a entero, e imprima el dtype final.

n_sin_dato = None

In [ ]:
if n_sin_dato is None:
    comprobar('T5', None)
else:
    comprobar('T5', [n_sin_dato, str(df['cod_municipio'].dtype)])

**Tu respuesta:** los `sin dato` de `cod_municipio` quedaron convertidos en 0, y el código DANE 0 no
existe. ¿Es aceptable? ¿Qué otra opción tenía, y por qué eligió esta?

*Tu respuesta:*

---

## Parte 4 · Duplicados

### Tarea 6 · Contar, ver y eliminar

**La pregunta.** ¿Cuántas filas están duplicadas, y cuántas quedan después de eliminarlas?

**El concepto.** Un duplicado no produce ningún error: infla los conteos y distorsiona los promedios
en silencio. Si Quibdó aparece dos veces en 2015, Quibdó pesa el doble en cualquier promedio que
calcule.

`duplicated()` devuelve una máscara —la misma idea de la clase 2— que marca `True` a partir de la
**segunda** aparición. `duplicated(keep=False)` marca **todas** las copias, incluida la primera: eso
es lo que sirve para **verlas** y confirmar que efectivamente son la misma fila.

**Un detalle que le va a cambiar el número:** ya rellenó nulos con la mediana. Dos filas que antes se
diferenciaban solo en que a una le faltaba un valor, ahora son idénticas. Puede que encuentre más
duplicados de los que había en el archivo original, y eso es correcto: **limpiar crea duplicados**.

**Los comandos.**

```python
df.duplicated().sum()                       # cuantos
df[df.duplicated(keep=False)]               # verlos todos, para confirmar
df = df.drop_duplicates()                   # eliminar, se queda con la primera
df.drop_duplicates(subset=['col1','col2'])  # cuando la llave es un subconjunto
```

**Lo que decide usted.** Si `drop_duplicates()` a secas es lo correcto aquí, o si debería comparar
solo algunas columnas. La pregunta previa siempre es la misma: **¿qué representa una fila en este
dataset?**

In [ ]:
# TU CÓDIGO AQUÍ
# 1. Guarde en n_duplicados cuántas filas están duplicadas.
# 2. Muéstrelas con keep=False, ordenadas, con las columnas ano, departamento y municipio.
# 3. Elimínelas y reasigne df. Imprima cuántas filas quedaron.

n_duplicados = None

In [ ]:
if n_duplicados is None:
    comprobar('T6', None)
else:
    comprobar('T6', [n_duplicados, len(df)])

**Tu respuesta:** en este dataset una fila representa un municipio en un año. Con esa unidad de
análisis, ¿un duplicado es siempre un error? Dé un ejemplo de un dataset donde dos filas idénticas
serían dos hechos reales distintos.

*Tu respuesta:*

---

## Parte 5 · Inconsistencias de texto

**Qué se practica aquí.** El problema más traicionero de los cinco: no produce ningún error **y
además parece que funciona**. El código corre, el resultado sale, y los números están repartidos
entre tres escrituras del mismo departamento.

`nunique()` cuenta cuántos valores distintos tiene una columna. Es el detector: usted sabe cuántos
debería haber, mira cuántos hay, y la diferencia es la suciedad.

### Tarea 7 · Estandarizar `departamento`

**La pregunta.** ¿Cuántos valores únicos tiene `departamento` antes y después de estandarizarla?

**El concepto.** Cuatro tratamientos, que se aplican encadenados porque cada uno devuelve una Series
sobre la que se puede volver a llamar `.str`:

1. `.str.strip()` quita los espacios de los extremos. Son invisibles al imprimir; por eso conviene
   imprimir los valores entre comillas.
2. `.str.upper()` unifica mayúsculas y minúsculas. Podría ser `.str.lower()`: da igual mientras sea
   consistente en todo el proyecto.
3. Las **tildes**: `NARIÑO` y `NARINO` son dos textos distintos para pandas. La receta estándar es la
   de abajo; se copia tal cual.
4. La **puntuación**, que ninguna receta general resuelve: hay que imprimir la lista, mirarla y
   decidir. No toda limpieza se automatiza.

**Los comandos.**

```python
df['columna'].nunique()
df['columna'] = df['columna'].str.strip().str.upper()
df['columna'] = (df['columna']
                 .str.normalize('NFKD')
                 .str.encode('ascii', 'ignore')
                 .str.decode('utf-8'))
df['columna'] = df['columna'].str.replace('CARACTER', '', regex=False)
```

**Lo que decide usted.** Cuántos departamentos **debería** haber (imprima la lista final y cuéntelos:
no son 33, este dataset no cubre el país entero), y qué puntuación sobra. Guarde el conteo de antes
en una variable **antes** de tocar la columna.

**Y una decisión que ya está tomada: `municipio` no se toca.** Compruebe su `nunique()`: los valores
ya están escritos de forma consistente. Estandarizar por reflejo una columna que no lo necesita es
tan mal criterio como no estandarizar la que sí.

In [ ]:
# TU CÓDIGO AQUÍ
# 1. Guarde en unicos_antes el nunique() de 'departamento'. Imprima también el de 'municipio'
#    y unos 20 valores de cada una, entre comillas, para ver el problema.
# 2. Estandarice SOLO 'departamento': espacios, mayúsculas, tildes y puntuación.
# 3. Imprima el nunique() final y la lista ordenada completa. Mírela.

unicos_antes = None

In [ ]:
if unicos_antes is None:
    comprobar('T7', None)
else:
    comprobar('T7', [unicos_antes, df['departamento'].nunique()])

### Tarea 8 · Los duplicados que acaba de crear

**La pregunta.** ¿Cuántos duplicados aparecieron **después** de estandarizar el texto, y cuántas
filas quedan al eliminarlos?

**El concepto.** `Antioquia` y `ANTIOQUIA` para el mismo municipio y el mismo año eran dos filas
distintas para pandas y una sola en la realidad. Al estandarizar, la realidad se hizo visible.

**Limpiar un problema crea otro.** La limpieza no es una lista de tareas que se tacha una vez: es
iterativa. De ahí sale la única regla de orden que no se negocia en este workflow: **los duplicados
se revisan siempre después de tocar el texto.**

**Los comandos.** Los mismos de la tarea 6.

**Lo que decide usted.** Nada nuevo de técnica. Lo que se practica aquí es el **reflejo** de volver a
verificar en vez de dar un paso por terminado.

In [ ]:
# TU CÓDIGO AQUÍ
# 1. Guarde en n_duplicados_nuevos cuántos duplicados hay ahora.
# 2. Elimínelos y reasigne df.
# 3. Imprima cuántas filas quedaron.

n_duplicados_nuevos = None

In [ ]:
if n_duplicados_nuevos is None:
    comprobar('T8', None)
else:
    comprobar('T8', [n_duplicados_nuevos, len(df)])

**Tu respuesta:** ¿por qué aparecieron duplicados que antes no existían? ¿Qué le dice eso sobre el
orden de los pasos de limpieza, y qué habría pasado si hubiera hecho las partes 4 y 5 al revés?

*Tu respuesta:*

---

## Parte 6 · Valores inválidos de dominio

**Qué se practica aquí.** El único de los cinco problemas que **no se puede detectar sin conocer el
dominio**. pandas no tiene idea de qué mide cada columna: solo usted puede decir qué valor es
imposible.

**La trampa, dicha una vez más porque es la que hunde este reto:** solo `bajo_peso_nacer`,
`partos_cesarea` y `partos_institucionales` son porcentajes acotados entre 0 y 100. Las columnas de
mortalidad son **tasas por mil o por cien mil**: `mortalidad_materna` de 422 significa 422 muertes
maternas por cada cien mil nacidos vivos, y es un dato perfectamente válido. Si las valida contra
0-100 va a "corregir" datos que estaban bien, y el destrozo no deja rastro.

Es exactamente el caso de `cobertura_neta` contra `cobertura_bruta` del demo: dos columnas que se ven
idénticas en un `describe()`, con reglas distintas.

### Tarea 9 · Detectar

**La pregunta.** ¿Cuántos valores están fuera del rango válido, sumando las tres columnas que sí son
porcentajes?

**El concepto.** Una máscara por condición y un conteo, como en la clase 2, pero aplicado a varias
columnas a la vez: `df[lista] < 0` devuelve un DataFrame de `True`/`False`, y `.sum()` sobre él da un
conteo por columna. Un `.sum()` más lo reduce al total.

**Los comandos.**

```python
columnas = ['col_a', 'col_b', 'col_c']
df[columnas].describe().loc[['min', 'max']]
(df[columnas] < 0).sum()        # conteo por columna
(df[columnas] > 100).sum()
int((df[columnas] < 0).sum().sum() + (df[columnas] > 100).sum().sum())
```

**Lo que decide usted.** **Cuáles son las tres columnas.** Es la única decisión de la tarea y es toda
la tarea: meter una columna de mortalidad en esa lista es el error que este reto existe para
provocar.

In [ ]:
# TU CÓDIGO AQUÍ
# 1. Defina columnas_porcentaje con las TRES columnas que son porcentajes.
# 2. Muestre su min y su max.
# 3. Guarde en n_invalidos cuántos valores están por debajo de 0 o por encima de 100,
#    sumando las tres columnas. Imprima también el desglose por columna.

n_invalidos = None

In [ ]:
comprobar('T9', n_invalidos)

### Tarea 10 · Decidir y corregir

**La pregunta.** Deje las tres columnas de porcentaje dentro del rango 0-100, **sin perder ninguna
fila**.

**El concepto.** Hay dos caminos defendibles y usted elige uno:

- **(a) Convertirlos en `NaN` y rellenar con la mediana.** Asume que el valor real era desconocido y
  que lo típico es la mejor estimación disponible.
- **(b) Recortarlos al rango válido:** los negativos a 0, los mayores de 100 a 100. Asume que el error
  fue de medición o digitación y que el valor real estaba cerca del extremo.

Hay un tercer camino que **no** es defendible aquí: borrar la fila. Un `partos_cesarea` imposible no
invalida la `mortalidad_infantil` de ese mismo municipio y año, que está perfectamente bien. Se
corrige el valor, no se tira la fila.

**Qué es `df.loc[condicion, 'columna'] = valor`:** la forma de modificar el **original**. Se lee como
una frase: "en las filas que cumplen esta condición, en esta columna, pon este valor". La alternativa
que no funciona es asignar sobre el resultado de un filtro (`df[condicion]['columna'] = valor`): eso
modifica una tabla nueva que se descarta enseguida, el original queda igual y **no aparece ninguna
advertencia**. El síntoma es el silencio.

**Los comandos.**

```python
invalidos = (df[col] < 0) | (df[col] > 100)
df.loc[invalidos, col] = np.nan            # camino (a)
df[col] = df[col].fillna(df[col].median())
df[col] = df[col].clip(0, 100)             # camino (b), en una linea
```

**Lo que decide usted.** Cuál de los dos caminos, y por qué. **Escríbalo abajo antes de programarlo.**
La comprobación acepta los dos: verifica que no queden valores fuera de rango y que no haya perdido
filas.

*Tu decisión y tu argumento:*

In [ ]:
# TU CÓDIGO AQUÍ
# 1. Aplique a las tres columnas de porcentaje el camino que justificó arriba.
# 2. Verifique: imprima el min y el max de las tres, y cuántos valores inválidos quedan.

In [ ]:
if 'columnas_porcentaje' not in globals():
    comprobar('T10', None)
else:
    restantes_invalidos = int((df[columnas_porcentaje] < 0).sum().sum()
                              + (df[columnas_porcentaje] > 100).sum().sum())
    comprobar('T10', [restantes_invalidos, len(df)])

### Las columnas de mortalidad: la que NO se corrige

**No las toque.** Ejecute la celda de abajo, mire los máximos, y responda.

In [ ]:
columnas_tasa = ['fecundidad_adolescente', 'mortalidad_fetal', 'mortalidad_general',
                 'mortalidad_infantil', 'mortalidad_materna', 'mortalidad_neonatal']

print(df[columnas_tasa].describe().loc[['min', 'max']].round(2).to_string())

**Tu respuesta:** `mortalidad_materna` tiene un máximo muy por encima de 100 y no lo corregimos. ¿Por
qué eso no es un error? ¿Qué habría pasado con el análisis si le hubiera aplicado la misma validación
que a `partos_cesarea`?

*Tu respuesta:*

---

## Parte 7 · Verificación final

**Qué cambia aquí.** Nada de técnica: no hay un solo comando que no haya usado ya. Lo que cambia es
que **el ensamblaje es suyo**. Le listamos los comandos disponibles; el orden en que se arman, no.

```python
df[df['columna'] == valor]                  df[['col1', 'col2']]
df.sort_values('columna', ascending=False)  df.head(5)
df['columna'].tolist()                      df['columna'].nunique()
df.isnull().sum().sum()                     df.duplicated().sum()
len(df)                                     df['columna'].dtype
```

Esta parte se empieza en el salón y se termina en casa.

### Antes y después

**La limpieza no está terminada hasta que se puede demostrar.** Para eso guardó `df_original` en el
paso 0. Compare las dos versiones en las seis dimensiones que tocó: filas, celdas vacías, duplicados,
departamentos únicos, y el tipo de `ano` y de `cod_municipio`.

Esta celda no se comprueba con `comprobar()`: la salida **es** la evidencia, y va en el entregable.

In [ ]:
# TU CÓDIGO AQUÍ
# Escriba una función resumen(datos, etiqueta) que imprima las seis cifras,
# y llámela dos veces: con df_original y con df.

### Tarea 11 · Una pregunta de verdad sobre datos limpios

**La pregunta.** ¿Cuáles son los **5 municipios** con mayor `mortalidad_infantil` del dataset?
Guarde en `top5_municipios` la **lista** de sus nombres, y muestre la tabla con departamento,
municipio, año y el indicador para poder interpretarla.

**El concepto.** Ningún comando nuevo: filtrar, ordenar y elegir columnas, todo de la clase 2 salvo
`sort_values`, que ordena por la columna que se le indique. Lo que se practica es el **ensamblaje**:
la pregunta viene en español y usted decide el orden de las operaciones.

Dos avisos: `sort_values` ordena de menor a mayor por defecto, que es lo contrario de lo que pide la
pregunta; y `municipio` no se estandarizó, así que los nombres van con tilde, tal como están en el
dato.

**Lo que decide usted.** El orden de las operaciones y el valor de `ascending`.

In [ ]:
# TU CÓDIGO AQUÍ
# 1. Ordene df por 'mortalidad_infantil' de mayor a menor y quédese con las 5 primeras filas.
# 2. Muestre la tabla con departamento, municipio, ano y mortalidad_infantil.
# 3. Guarde en top5_municipios la lista de los nombres de municipio.

top5_municipios = None

In [ ]:
comprobar('T11', top5_municipios)

**Tu respuesta:** mire la lista. ¿Hay municipios que se repiten? ¿Qué le dice eso: que hubo un año
malo, o que hay un problema estructural? Y una tercera frase incómoda: **¿cuánto confía en este
ranking**, sabiendo que usted mismo rellenó cerca del 11% de esa columna con la mediana?

*Tu respuesta:*

---

## Punto de control

Ejecute la celda de abajo para ver cuántas de las once tareas quedaron correctas.

Si alguna sigue pendiente, no pase de largo: la clase 4 arranca dando por sabido que usted puede
dejar un dataset limpio. Si está en el salón, levante la mano ahora, que el profesor está aquí para
eso.

In [ ]:
resumen_puntos_de_control()

---

## Reflexión (en casa)

Responda en español, dos o tres frases por pregunta. Esto no se comprueba con `comprobar()` y es lo
que más pesa.

**1. ¿Cuál de los 5 problemas fue el más difícil de detectar? ¿Por qué?**

*Tu respuesta:*

**2. ¿Alguna de sus decisiones de limpieza podría cambiar la conclusión de un análisis posterior?
¿Cuál y cómo?**

*Tu respuesta:*

**3. Piense en el dataset que su equipo eligió para el proyecto. ¿Cuáles de estos 5 problemas espera
encontrar? ¿Cómo va a verificarlo?**

*Tu respuesta:*

---

## Opcional · Solo si terminó todo

No se comprueba y no entra en la retroalimentación.

**A.** Exporte el dataset limpio: `df.to_csv('indicadores_salud_limpio.csv', index=False)`.

**B.** Escriba una función `diagnostico(datos)` que reciba cualquier DataFrame e imprima el resumen
de los 5 problemas: forma, tipos, porcentaje de nulos por columna, duplicados y valores únicos de las
columnas de texto. Es la parte automatizable del oficio, y la va a agradecer en el Momento 1.

**C.** Vuelva a la tarea 10 y aplique el **otro** camino sobre `df_original`. ¿Cambia el ranking de
la tarea 11? Si cambia, acaba de encontrar el mejor argumento del semestre a favor de justificar las
decisiones por escrito.

In [ ]:
# TU CÓDIGO AQUÍ (opcional)

---

## Antes de entregar

1. **Kernel → Restart and Run All.** De arriba a abajo, una sola vez. Si algo revienta, arréglelo: un
   cuaderno que no corre completo le pone techo a la dimensión Hacer.
2. Verifique que **cada decisión** tiene su justificación escrita: las de la parte 2, la de
   `cod_municipio`, la de duplicados, la de la parte 6 y la de por qué no tocó mortalidad. En
   limpieza de datos la justificación pesa más que el código.
3. Ejecute el punto de control y deje la salida a la vista.
4. Guarde como `clase03_reto_APELLIDO.ipynb` y súbalo al aula virtual, antes del inicio de la clase 4.